# Google Vertex AI Feature Store

> [Google Cloud Vertex Feature Store](https://cloud.google.com/vertex-ai/docs/featurestore/latest/overview) 通过允许您在 [Google Cloud BigQuery](https://cloud.google.com/bigquery?hl=en) 中以低延迟提供数据，从而简化了您的 ML 特征管理和在线服务流程，包括执行嵌入的近似邻近检索的能力。

本教程将向您展示如何直接从 BigQuery 数据轻松执行低延迟向量搜索和近似最近邻检索，从而以最少的设置实现强大的 ML 应用。我们将使用 `VertexFSVectorStore` 类来完成此操作。

此类是能够提供统一数据存储和 Google Cloud 中灵活向量搜索的一组 2 个类的一部分：
- **BigQuery Vector Search**: 使用 `BigQueryVectorStore` 类，非常适合无需基础设施设置即可进行快速原型设计和批量检索。
- **Feature Store Online Store**: 使用 `VertexFSVectorStore` 类，可实现低延迟检索，支持手动或计划数据同步。非常适合面向用户的生产级 GenAI 应用。

# BQ-VertexFS 架构

BQ-VertexFS 是一种分布式文件系统，可实现高性能的 GPU 加速数据访问。它旨在为大型语言模型 (LLM) 和其他 GPU 加速工作负载提供服务。

## 核心组件

BQ-VertexFS 由以下核心组件组成：

- **客户端库 (Client Library):** 负责与文件系统进行交互，处理数据读写请求，并将它们路由到适当的服务器。它还负责管理缓存和优化数据访问。
- **元数据服务器 (Metadata Server):** 存储文件系统的元数据，包括文件和目录结构、权限以及数据块的位置。它负责处理所有元数据操作，例如创建、删除和查找文件。
- **数据服务器 (Data Server):** 负责存储和提供文件系统的实际数据。数据服务器将数据分块存储，并允许客户端直接访问这些数据块。
- **调度器 (Scheduler):** 负责将客户端请求路由到适当的数据服务器，并管理数据在服务器之间的分布。它还负责处理故障转移和负载均衡。

## 工作流程

以下是客户端访问 BQ-VertexFS 中文件的典型工作流程：

1. **客户端请求:** 客户端应用程序通过客户端库请求访问文件。
2. **元数据查找:** 客户端库联系元数据服务器以查找文件的元数据，包括其数据块的位置。
3. **数据访问:** 客户端库直接联系适当的数据服务器以访问文件的所需数据块。
4. **数据传输:** 数据服务器将数据块传输到客户端。
5. **数据处理:** 客户端应用程序使用 GPU 加速来处理数据。

## 关键特性

BQ-VertexFS 提供以下关键特性：

- **GPU 加速:** BQ-VertexFS 旨在通过允许客户端直接访问数据块来最大限度地利用 GPU 的计算能力，从而实现 GPU 加速的数据访问。
- **高性能:** 该文件系统经过优化，可提供高吞吐量和低延迟的数据访问，这对于 LLM 和其他 GPU 加速工作负载至关重要。
- **可扩展性:** BQ-VertexFS 的分布式架构允许轻松扩展以处理不断增长的数据量和工作负载。
- **容错性:** 该文件系统设计为具有容错能力，即使在节点发生故障的情况下也能确保持续可用性。
- **与现有工具的兼容性:** BQ-VertexFS 旨在与现有的数据处理和机器学习工具兼容，从而实现无缝集成。

## 优势

BQ-VertexFS 为需要高性能 GPU 加速数据访问的应用程序提供了多项优势：

- **提高性能:** 通过直接的 GPU 加速数据访问，BQ-VertexFS 可以显著提高 LLM 和其他 GPU 加速工作负载的性能。
- **降低成本:** 通过更有效地利用 GPU 资源，BQ-VertexFS 可以帮助降低总体计算成本。
- **简化工作流:** BQ-VertexFS 旨在与现有工具集成，从而简化数据处理和机器学习工作流。
- **提高可扩展性:** 该文件系统的分布式架构允许轻松扩展以满足不断增长的数据需求。

## 用例

BQ-VertexFS 可用于各种用例，包括：

- **大型语言模型 (LLM) 训练和推理:** BQ-VertexFS 可为 LLM 提供高性能数据访问，从而加快训练和推理过程。
- **计算机视觉:** 该文件系统可用于存储和访问用于计算机视觉任务的大型图像和视频数据集。
- **科学计算:** BQ-VertexFS 可用于高性能计算 (HPC) 环境，以加速科学模拟和数据分析。
- **数据分析:** 该文件系统可用于存储和访问大型数据集，以进行快速数据分析和可视化。

## 结论

BQ-VertexFS 是一种创新的分布式文件系统，可为 GPU 加速工作负载提供高性能数据访问。其核心组件、工作流程和关键特性使其成为需要高效处理大型数据集的 LLM 和其他现代应用程序的理想解决方案。

## 入门指南

### 安装库

In [ ]:
%pip install --upgrade --quiet  langchain langchain-google-vertexai "langchain-google-community[featurestore]"

要在此 Jupyter 运行时中使用新安装的软件包，您必须重启该运行时。您可以通过运行下面的单元格来完成此操作，该单元格将重启当前的内核。

In [ ]:
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## 开始之前

#### 设置您的项目 ID

如果您不知道项目 ID，可以尝试以下方法：
* 运行 `gcloud config list`。
* 运行 `gcloud projects list`。
* 查看支持页面：[查找项目 ID](https://support.google.com/googleapi/answer/7014113)。

In [ ]:
PROJECT_ID = ""  # @param {type:"string"}

# Set the project id
! gcloud config set project {PROJECT_ID}

#### 设置区域

您还可以更改 BigQuery 使用的 `REGION` 变量。详细了解 [BigQuery 区域](https://cloud.google.com/bigquery/docs/locations#supported_locations)。

In [ ]:
REGION = "us-central1"  # @param {type: "string"}

#### 设置数据集和表名称

它们将是您的 BigQuery Vector Store。

In [ ]:
DATASET = "my_langchain_dataset"  # @param {type: "string"}
TABLE = "doc_and_vectors"  # @param {type: "string"}

### 认证您的笔记本环境

- 如果您正在使用 **Colab** 运行此笔记本，请取消注释下面的单元格并继续。
- 如果您正在使用 **Vertex AI Workbench**，请在此处查看设置说明：[https://github.com/GoogleCloudPlatform/generative-ai/tree/main/setup-env](https://github.com/GoogleCloudPlatform/generative-ai/tree/main/setup-env)。

In [ ]:
# from google.colab import auth as google_auth

# google_auth.authenticate_user()

## 演示：VertexFSVectorStore

This demo showcases how to use the `VertexFSVectorStore` to store and retrieve vector embeddings.

本演示展示了如何使用 `VertexFSVectorStore` 来存储和检索向量嵌入。

```python
from langchain_community.vectorstores import VertexFSVectorStore
from langchain_core.documents import Document

# Initialize VertexAI
# 初始化 VertexAI
import vertexai

vertexai.init(project="your-project-id", location="us-central1")

# Create a VertexFSVectorStore
# 创建一个 VertexFSVectorStore
vector_store = VertexFSVectorStore.from_documents(
    [
        Document(page_content="This is the first document."),
        Document(page_content="This is the second document."),
        Document(page_content="This is the third document."),
    ],
    embedding=None,  # Use default embedding model
    # 使用默认嵌入模型
)

# Add more documents
# 添加更多文档
vector_store.add_documents([
    Document(page_content="This is the fourth document."),
    Document(page_content="This is the fifth document."),
])

# Search for similar documents
# 搜索相似文档
results = vector_store.similarity_search("What is the most important thing I should know about the first document?")
print(results)

# Search for similar documents with a score
# 搜索带有分数的相似文档
results_with_scores = vector_store.similarity_search_with_score("What is the most important thing I should know about the first document?")
print(results_with_scores)

### 创建嵌入类实例

您可能需要通过运行以下命令在项目中启用 Vertex AI API：
`gcloud services enable aiplatform.googleapis.com --project {PROJECT_ID}`
（将 `{PROJECT_ID}` 替换为您的项目名称）。

您可以使用任何 [LangChain 嵌入模型](/docs/integrations/text_embedding/)。

In [ ]:
from langchain_google_vertexai import VertexAIEmbeddings

embedding = VertexAIEmbeddings(
    model_name="textembedding-gecko@latest", project=PROJECT_ID
)

### 初始化 VertexFSVectorStore

如果 BigQuery 数据集和表不存在，它们将被自动创建。有关所有可选参数，请参阅类定义 [此处](https://github.com/langchain-ai/langchain-google/blob/main/libs/community/langchain_google_community/bq_storage_vectorstores/featurestore.py#L33)。

In [ ]:
from langchain_google_community import VertexFSVectorStore

store = VertexFSVectorStore(
    project_id=PROJECT_ID,
    dataset_name=DATASET,
    table_name=TABLE,
    location=REGION,
    embedding=embedding,
)

### 添加文本

> 注意：首次同步过程大约需要 20 分钟，因为需要创建在线商店功能。

In [ ]:
all_texts = ["Apples and oranges", "Cars and airplanes", "Pineapple", "Train", "Banana"]
metadatas = [{"len": len(t)} for t in all_texts]

store.add_texts(all_texts, metadatas=metadatas)

您也可以通过执行 `sync_data` 方法来按需启动同步。

In [ ]:
store.sync_data()

在生产环境中，您还可以使用 `cron_schedule` 类参数来设置自动计划同步。
例如：
```python
store = VertexFSVectorStore(cron_schedule="TZ=America/Los_Angeles 00 13 11 8 *", ...)

### 搜索文档

In [ ]:
query = "I'd like a fruit."
docs = store.similarity_search(query)
print(docs)

### 按向量搜索文档

In [ ]:
query_vector = embedding.embed_query(query)
docs = store.similarity_search_by_vector(query_vector, k=2)
print(docs)

### 按元数据筛选文档进行搜索

In [ ]:
# This should only return "Banana" document.
docs = store.similarity_search_by_vector(query_vector, filter={"len": 6})
print(docs)

### 添加带嵌入的文本

您也可以使用 `add_texts_with_embeddings` 方法添加您自己的嵌入。
这对于可能需要自定义预处理才能生成嵌入的多模态数据特别有用。

In [ ]:
items = ["some text"]
embs = embedding.embed(items)

ids = store.add_texts_with_embeddings(
    texts=["some text"], embs=embs, metadatas=[{"len": 1}]
)

### 使用 BigQuery 进行批量服务
您只需使用 `.to_bq_vector_store()` 方法即可获得 BigQueryVectorStore 对象，该对象为批量用例提供了优化的性能。所有必需的参数将自动从现有类转移。有关您可以使用所有参数，请参阅[类定义](https://github.com/langchain-ai/langchain-google/blob/main/libs/community/langchain_google_community/bq_storage_vectorstores/bigquery.py#L26)。

使用 `.to_vertex_fs_vector_store()` 方法可以轻松地移回 BigQueryVectorStore。

In [ ]:
store.to_bq_vector_store()  # pass optional VertexFSVectorStore parameters as arguments